In [1]:
# ! pip install ultralytics

In [1]:
from utils.multicamera_tools import parse_camera_xml, triangulate_poses
from utils.video_tools import get_camera_calibration_files, get_video_files
from scripts.frame_iterator import video_frame_iterator
from scripts.parsers import parse_sequences as parse_sequence_info
import numpy as np
import bvhio
import warnings

warnings.filterwarnings('ignore')

file_path = 'gait3d\\ListOfSequences.txt'
sequences = parse_sequence_info(file_path)

In [2]:
from ultralytics import YOLO

# model = YOLO("yolo11n-pose.pt")
model = YOLO("yolo11x-pose.pt")

In [3]:
selected_joint_names = {
    5: 'lhumerus',
    6: 'rhumerus',
    11: 'lfemur',
    12: 'rfemur',
    13: 'ltibia',
    14: 'rtibia',
    15: 'lfoot',
    16: 'rfoot'
}

selected_joint_names

{5: 'lhumerus',
 6: 'rhumerus',
 11: 'lfemur',
 12: 'rfemur',
 13: 'ltibia',
 14: 'rtibia',
 15: 'lfoot',
 16: 'rfoot'}

In [20]:
FRAME_WIDTH = 960
FRAME_HEIGHT = 540

for i in range(1, 5):
    results = model.predict(
        source=f'./gait3d/Sequences/p5s1/Images/c{i}_0195.avi',
        show=False, # do not display during processing
        save=False, # save annotated video
        project='sample_vids',
        name='yolo11', 
        # exist_ok=True,
        verbose=False, 
        stream=True
    )

    for result in [next(results)]:
        # print(result.keypoints)
        
        if not len(result.keypoints.xyn) == 1:
            xy_n == [[0, 0] for _ in range(17)]
        xy_n = result.keypoints.xyn[0].cpu().numpy()
        xy_abs = xy_n * [FRAME_WIDTH, FRAME_HEIGHT]
    
        # print(f"{xy_n = }")
        print("-------------------------------------------------------")
        for item in zip(xy_abs, result.keypoints.xy.cpu().numpy()[0], result.keypoints.conf.cpu().numpy()[0]):
        # print(f"{xy_abs = }")
        # print(f"{result.keypoints.xy.cpu().numpy() = }")
        # print(f"{result.keypoints.conf.cpu().numpy() = }")
            print(item)
        break

-------------------------------------------------------
(array([     803.95,      138.25]), array([     803.95,      138.25], dtype=float32), 0.87566894)
(array([     808.43,      133.89]), array([     808.43,      133.89], dtype=float32), 0.8741964)
(array([          0,           0]), array([          0,           0], dtype=float32), 0.1706694)
(array([      823.1,      137.34]), array([      823.1,      137.34], dtype=float32), 0.92739254)
(array([          0,           0]), array([          0,           0], dtype=float32), 0.017775485)
(array([     837.43,      169.79]), array([     837.43,      169.79], dtype=float32), 0.99602973)
(array([     805.35,       168.8]), array([     805.35,       168.8], dtype=float32), 0.99491185)
(array([     837.72,      212.14]), array([     837.72,      212.14], dtype=float32), 0.99270576)
(array([     798.42,      206.04]), array([     798.42,      206.04], dtype=float32), 0.98483187)
(array([     818.47,      245.71]), array([     818.47,      24

In [6]:
FRAME_WIDTH = 960
FRAME_HEIGHT = 540

for result in results:
    xy_n = result.keypoints.xy_n[0].cpu().numpy()  # normalized
    xy_abs = xy_n * [FRAME_WIDTH, FRAME_HEIGHT]

    print(f"{xy_n = }")
    print(f"{xy_abs }")
    break

In [58]:
VIDEO_FPS = 25
MOCAP_FPS = 100
FRAME_TIME = 1000/VIDEO_FPS
FRAME_WIDTH = 960
FRAME_HEIGHT = 540
YOLO_LANDMARKS_NUM = 17

yolo_selection = {}
yolo_triangulation = {}

for seq_key in list(sequences.keys()):
    print(seq_key, end=" | ")
    if sequences[seq_key]['MoCap_data']:
        video_files = get_video_files(seq_key)
        max_frames = sequences[seq_key]['number_of_frames']
                
        camera_files_paths = get_camera_calibration_files(seq_key)
        cameras_params = [parse_camera_xml(camera_path) for camera_path in camera_files_paths]
    
        predicted_for_seq = {f"c{i+1}": {} for i in range(4)}
        prediction_confidence_for_seq = {f"c{i+1}": {} for i in range(4)}
        combined_triangulation_results = []
        
        for c_idx, c_file in enumerate(video_files):
            # print(c_idx + 1, c_file)
            results = model.predict(
                source=c_file,
                show=False, # do not display during processing
                save=False, # do not save annotated video
                project='sample_vids',
                name='yolo11', 
                verbose=False, 
                stream=True
            )

            
            for f_idx, result in enumerate(results):
                if len(result.keypoints.xy) == 1 and len(result.keypoints.xy[0] == YOLO_LANDMARKS_NUM) and result.keypoints.conf is not None:
                    results_xy = result.keypoints.xy.cpu().numpy().tolist()
                    results_conf = result.keypoints.conf.cpu().numpy().tolist()
                    xy_n = results_xy[0]
                    xy_confidence = results_conf[0]

                else:
                    xy_n == [[None, None] for _ in range(YOLO_LANDMARKS_NUM)]
                    xy_confidence = [0 for _ in range(YOLO_LANDMARKS_NUM)]

                predicted_for_seq[f"c{c_idx+1}"][f_idx] = xy_n
                prediction_confidence_for_seq[f"c{c_idx+1}"][f_idx] = xy_confidence

        yolo_selection[seq_key] = predicted_for_seq

        for f_idx in range(max_frames):
            triangulation_result = []
            for landmark_idx in range(YOLO_LANDMARKS_NUM):
                min_conf_camera = prediction_confidence_for_seq["c1"][f_idx][landmark_idx]
                min_conf_camera_idx = 0
                for camera_idx in range(1,4):
                    if (curr_conf := prediction_confidence_for_seq[f"c{camera_idx+1}"][f_idx][landmark_idx]) < min_conf_camera:
                        min_conf_camera = curr_conf
                        min_conf_camera_idx = camera_idx

                selected_cameras_idx = [i for i in range(4) if i != min_conf_camera_idx]
                assert len(selected_cameras_idx) == 3
                selected_cameras_params = [cameras_params[camera_i] for camera_i in selected_cameras_idx]
                found_2d_points = np.array([[predicted_for_seq[f"c{camera_idx+1}"][f_idx][landmark_idx]] for camera_idx in selected_cameras_idx])
                landmark_triangulation_result = triangulate_poses(selected_cameras_params, found_2d_points)
                triangulation_result.append(landmark_triangulation_result[0][0].tolist())
            
            combined_triangulation_results.append(triangulation_result)

        yolo_triangulation[seq_key] = combined_triangulation_results


p1s1 | p1s2 | p1s3 | p1s4 | p2s1 | p2s2 | p2s3 | p2s4 | p3s1 | p3s2 | p3s3 | p3s4 | p4s1 | p4s2 | p4s3 | p4s4 | p5s1 | p5s2 | p5s3 | p5s4 | p6s1 | p6s2 | p6s3 | p6s4 | p7s1 | p7s2 | p7s3 | p7s4 | p8s1 | p8s2 | p8s3 | p8s4 | p9s1 | p9s2 | p9s3 | p9s4 | p10s1 | p10s2 | p10s3 | p10s4 | p11s1 | p11s2 | p11s3 | p11s4 | p12s1 | p12s2 | p12s3 | p12s4 | p13s1 | p13s2 | p13s3 | p13s4 | p14s1 | p14s2 | p14s3 | p14s4 | p15s1 | p15s2 | p15s3 | p15s4 | p16s1 | p16s2 | p16s3 | p16s4 | p17s1 | p17s2 | p17s3 | p17s4 | p18s1 | p18s2 | p18s3 | p18s4 | p19s1 | p19s2 | p19s3 | p19s4 | p20s1 | p20s2 | p20s3 | p20s4 | p21s1 | p21s2 | p21s3 | p21s4 | p22s1 | p22s2 | p22s3 | p22s4 | p23s1 | p23s2 | p23s3 | p23s4 | p24s1 | p24s2 | p24s3 | p24s4 | p25s1 | p25s2 | p25s3 | p25s4 | p26s1 | p26s2 | p26s3 | p26s4 | p26s5 | p26s6 | p26s7 | p26s8 | p26s9 | p26s10 | p27s1 | p27s2 | p27s3 | p27s4 | p27s5 | p27s6 | p27s7 | p27s8 | p27s9 | p27s10 | p28s1 | p28s2 | p28s3 | p28s4 | p28s5 | p28s6 | p28s7 | p28s8 | p28s9 | p2

In [59]:
import json

with open("./datasets/yolo/triangulation_v2_best_cameras.json", "w") as f:
    json.dump(yolo_triangulation, f, indent=4)